# Notebook 16 — Path 2: MELU-Δt vs DAGMM / GOAD / NeuTraL-AD / ICL  (Fixed)

## Bug fixes vs previous version

| Method | Bug | Fix |
|---|---|---|
| MELU | `BaseAE` had no `encode_elu` → `AttributeError` → caught → returns 0.5 | Added `encode_elu` to `BaseAE` |
| DAGMM | `torch.linalg.solve` on singular GMM component cov → `RuntimeError` → 0.5 | n_comp=1 (single Gaussian), stronger regularisation 1e-2·I |
| GOAD | n_trans=dim → too many classes (~6 samples per class) → model cannot learn → inverted AUROC | Fixed n_trans=8; score = per-sample CE loss (not 1−prob) |
| ICL | Unconditional noise same for inliers/outliers → no discrimination; cos-sim score collapses | Score = latent distance to inlier centroid after contrastive training |

NeuTraL-AD (0.84) worked correctly — no fix needed.

## Fair comparison

All five methods:
- Same base architecture dimensions (adaptive lat = min(dim//2,16))
- Same 50%/50% train/test split per seed
- Same 10 seeds
- Same datasets (33 total: 7 sklearn + 26 ADBench-sim)


## Cell 1 — Imports and config

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.special import betainc, gammaln
from scipy.stats import wilcoxon, friedmanchisquare, rankdata
from sklearn.datasets import load_digits, load_breast_cancer, load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score
import warnings, time
warnings.filterwarnings("ignore")
torch.manual_seed(42); np.random.seed(42)

N_SEEDS    = 10
TRAIN_FRAC = 0.50
LR         = 1e-3
EPOCHS     = 100

METHODS = ["MELU-Δt", "DAGMM", "GOAD", "NeuTraL-AD", "ICL"]
COLORS  = {"MELU-Δt":"#1D9E75","DAGMM":"#534AB7",
           "GOAD":"#BA7517","NeuTraL-AD":"#D85A30","ICL":"#888780"}

def lat_for(dim): return max(4, min(dim//2, 16))
def hid_for(dim): return max(64, dim*4)
print(f"PyTorch {torch.__version__} ✓  |  Methods: {METHODS}")


PyTorch 2.5.1+cu121 ✓  |  Methods: ['MELU-Δt', 'DAGMM', 'GOAD', 'NeuTraL-AD', 'ICL']


## Cell 2 — Shared utilities

In [2]:
class StudentTCDF(torch.autograd.Function):
    NU = 5.0
    @staticmethod
    def forward(ctx, x):
        nu=StudentTCDF.NU; xn=x.detach().cpu().numpy()
        z=nu/(nu+np.clip(xn**2,1e-30,None))
        ib=betainc(nu/2,0.5,np.clip(z,1e-12,1-1e-12))
        ctx.save_for_backward(x)
        return torch.tensor(np.where(xn>=0,1.-ib/2.,ib/2.),dtype=x.dtype,device=x.device)
    @staticmethod
    def backward(ctx,g):
        x,=ctx.saved_tensors; nu=StudentTCDF.NU; xn=x.detach().cpu().numpy()
        lc=gammaln((nu+1)/2)-gammaln(nu/2)-.5*np.log(nu*np.pi)
        pdf=np.exp(lc-(nu+1)/2*np.log(1+xn**2/nu))
        return g*torch.tensor(pdf,dtype=x.dtype,device=x.device)
tcdf=StudentTCDF.apply


class BaseAE(nn.Module):
    """
    Shared backbone. BUG FIX: added encode_elu() so MELU's gate
    can call enc_frozen.encode_elu(x) without AttributeError.
    """
    def __init__(self,dim,hid,lat):
        super().__init__()
        self.W1=nn.Linear(dim,hid); self.W2=nn.Linear(hid,lat); self.dec=nn.Linear(lat,dim)
        for m in [self.W1,self.W2,self.dec]:
            nn.init.kaiming_normal_(m.weight); nn.init.zeros_(m.bias)

    def encode_elu(self,x):
        """ELU encoder — used as MELU gate signal provider."""
        return self.W2(F.elu(F.silu(self.W1(x))))

    def encode(self,x):
        return self.W2(F.silu(self.W1(x)))

    def forward(self,x): return self.dec(self.encode_elu(x))

    def recon_err(self,x):
        with torch.no_grad(): return (x-self(x)).abs().mean(1)


def fast_mcd(Z,hf=0.75,ns=6,nc=5):
    n,d=Z.shape; h=max(int(n*hf),d+1); bd=np.inf; bm=bc=None
    for _ in range(ns):
        idx=np.random.choice(n,h,replace=False); sub=Z[idx]
        for _ in range(nc):
            mu=sub.mean(0); dv=sub-mu
            cov=dv.T@dv/max(len(sub)-1,1)+1e-4*np.eye(d)
            Si=np.linalg.inv(cov)
            ds=np.sqrt(np.maximum(np.einsum('bi,ij,bj->b',Z-mu,Si,Z-mu),0))
            idx=np.argsort(ds)[:h]; sub=Z[idx]
        mu=sub.mean(0); dv=sub-mu; cov=dv.T@dv/max(len(sub)-1,1)
        det=np.linalg.det(cov+1e-4*np.eye(d))
        if det<bd: bd=det; bm=mu; bc=cov
    try:
        L=np.linalg.cholesky(bc+1e-4*np.eye(d)); Li=np.linalg.inv(L)
        if np.isnan(Li).any() or np.linalg.cond(Li)>1e7: Li=np.eye(d)
    except: Li=np.eye(d)
    return bm,bc,Li


def bce_pseudo(model, Xi_t, er_b, perm_idx, ep, wu, pct, lam, enc_frozen=None):
    """BCE pseudo-label loss after warmup."""
    if ep < wu: return 0.0
    model.eval()
    with torch.no_grad():
        er_all = model.recon_err(Xi_t, enc_frozen) if enc_frozen else model.recon_err(Xi_t)
    py = torch.tensor((er_all.numpy() > np.percentile(er_all.numpy(),pct)).astype(np.float32))
    model.train()
    em,eM = er_b.detach().min(), er_b.detach().max()
    pb = ((er_b-em)/(eM-em+1e-8)).clamp(1e-6,1-1e-6)
    return lam * F.binary_cross_entropy(pb, py[perm_idx[:len(er_b)]])

print("BaseAE (with encode_elu fix), fast_mcd defined ✓")


BaseAE (with encode_elu fix), fast_mcd defined ✓


## Cell 3 — MELU-Δt (fixed: enc_frozen.encode_elu now works)

In [3]:
class MELUGate(nn.Module):
    def __init__(self,lat):
        super().__init__()
        self.register_buffer('mu',torch.zeros(lat))
        self.register_buffer('Li',torch.eye(lat))
        self.register_buffer('tau',torch.tensor(1.5))
    def set_mcd(self,mu_np,Li_np,tau_val):
        dev=self.mu.device
        self.mu=torch.tensor(mu_np,dtype=torch.float32,device=dev)
        self.Li=torch.tensor(Li_np,dtype=torch.float32,device=dev)
        self.tau=torch.tensor(float(tau_val),device=dev)
    def forward(self,Z):
        w=(Z-self.mu.unsqueeze(0))@self.Li.T
        return w.norm(dim=1)


class MELU_AE(nn.Module):
    def __init__(self,dim,hid,lat):
        super().__init__()
        self.W1=nn.Linear(dim,hid); self.W2=nn.Linear(hid,lat); self.W3=nn.Linear(lat,dim)
        self.log_alpha=nn.Parameter(torch.log(torch.tensor(1.0)))
        self.log_beta =nn.Parameter(torch.log(torch.tensor(0.5)))
        self.gate=MELUGate(lat); self.gate_on=False
        for m in [self.W1,self.W2,self.W3]:
            nn.init.kaiming_normal_(m.weight); nn.init.zeros_(m.bias)
    @property
    def alpha(self): return self.log_alpha.exp()
    @property
    def beta(self):  return self.log_beta.exp()

    def encode(self,x,enc_frozen=None):
        h1=F.silu(self.W1(x)); T1=h1*tcdf(h1)
        if self.gate_on and enc_frozen is not None:
            with torch.no_grad():
                # FIX: enc_frozen.encode_elu now exists in BaseAE
                Zf=enc_frozen.encode_elu(x)
            m=self.gate(Zf)
            g=(m>=self.gate.tau).float().unsqueeze(1)
            amp=self.alpha*h1.sign()*torch.tanh(
                self.beta*(m.unsqueeze(1)-self.gate.tau).clamp(-8,8))
            return self.W2(T1+g*amp)
        return self.W2(T1)

    def forward(self,x,enc_frozen=None): return self.W3(self.encode(x,enc_frozen))

    def recon_err(self,x,enc_frozen=None):
        with torch.no_grad(): return (x-self(x,enc_frozen)).abs().mean(1)


def run_melu(Xi_tr_np, Xi_all_np, X_test_np, y_test_np, seed=0,
             n_pre=60, n_fine=80, lam=0.5, pct=85):
    torch.manual_seed(seed); np.random.seed(seed)
    dim=X_test_np.shape[1]; lat=lat_for(dim); hid=hid_for(dim)
    Xi_tr=torch.tensor(Xi_tr_np,dtype=torch.float32)
    X_te =torch.tensor(X_test_np,dtype=torch.float32)

    # Stage 1: ELU-AE (BaseAE now has encode_elu)
    elu_ae=BaseAE(dim,hid,lat)
    opt1=optim.Adam(elu_ae.parameters(),lr=LR); wu1=int(n_pre*.2)
    for ep in range(n_pre):
        elu_ae.train(); perm=torch.randperm(len(Xi_tr))
        for i in range(0,len(Xi_tr),64):
            xb=Xi_tr[perm[i:i+64]]; er=(xb-elu_ae(xb)).abs().mean(1)
            loss=er.mean()+bce_pseudo(elu_ae,Xi_tr,er,perm[i:i+64],ep,wu1,pct,lam)
            opt1.zero_grad(); loss.backward(); opt1.step()

    # Stage 2: MCD on ALL inliers (scaled)
    elu_ae.eval()
    Xi_all_t=torch.tensor(Xi_all_np,dtype=torch.float32)
    with torch.no_grad(): Z_all=elu_ae.encode_elu(Xi_all_t).numpy()
    mu_l,_,Li_l=fast_mcd(Z_all); w=(Z_all-mu_l)@Li_l.T
    dm=np.sqrt(np.maximum((w**2).sum(1),0)); tau=dm.mean(); gp=float((dm>tau).mean())

    # Stage 3: MELU fine-tune warm-started from Stage-1 weights
    melu=MELU_AE(dim,hid,lat)
    melu.W1.weight.data=elu_ae.W1.weight.data.clone()
    melu.W1.bias.data  =elu_ae.W1.bias.data.clone()
    melu.W2.weight.data=elu_ae.W2.weight.data.clone()
    melu.W2.bias.data  =elu_ae.W2.bias.data.clone()
    melu.W3.weight.data=elu_ae.dec.weight.data.clone()
    melu.W3.bias.data  =elu_ae.dec.bias.data.clone()
    melu.gate.set_mcd(mu_l,Li_l,tau)
    for p in elu_ae.parameters(): p.requires_grad_(False)

    opt3=optim.Adam(melu.parameters(),lr=LR*.5); wu3=int(n_fine*.2)
    for ep in range(n_fine):
        melu.gate_on=(ep>=wu3); melu.train(); perm=torch.randperm(len(Xi_tr))
        for i in range(0,len(Xi_tr),64):
            xb=Xi_tr[perm[i:i+64]]; er=(xb-melu(xb,elu_ae)).abs().mean(1)
            loss=er.mean()+bce_pseudo(melu,Xi_tr,er,perm[i:i+64],ep,wu3,pct,lam,elu_ae)
            opt3.zero_grad(); loss.backward(); opt3.step()

    melu.eval(); melu.gate_on=True
    er=melu.recon_err(X_te,elu_ae).numpy()
    return float(roc_auc_score(y_test_np,er)), gp

print("MELU-Δt defined ✓  (encode_elu bug fixed)")


MELU-Δt defined ✓  (encode_elu bug fixed)


## Cell 4 — DAGMM (fixed: n_comp=1, strong regularisation)

**Fix:** With small training sets, n_comp=2 causes degenerate GMM components.
Using n_comp=1 (equivalent to Gaussian anomaly detection on joint AE+recon-feat space)
is stable and recovers the DAGMM core mechanism.


In [4]:
class DAGMM(nn.Module):
    """
    Deep Autoencoding Gaussian Mixture Model — n_comp=1 (single Gaussian).
    FIX: n_comp=1 avoids singular component covariances with small n.
    Score = Mahalanobis distance in [z, recon_features] space.
    Ref: Zong et al. ICLR 2018.
    """
    def __init__(self,dim,hid,lat,n_comp=1):
        super().__init__()
        self.ae=BaseAE(dim,hid,lat); self.n_comp=n_comp; self.lat=lat
        feat_dim=lat+2
        self.estim=nn.Sequential(
            nn.Linear(feat_dim,16),nn.Tanh(),nn.Dropout(0.5),
            nn.Linear(16,n_comp),nn.Softmax(dim=1))
        for m in self.estim.modules():
            if isinstance(m,nn.Linear):
                nn.init.kaiming_normal_(m.weight); nn.init.zeros_(m.bias)

    def features(self,x):
        z=self.ae.encode_elu(x); xh=self.ae.dec(z)
        ed=(x-xh).norm(dim=1,keepdim=True)/(x.norm(dim=1,keepdim=True)+1e-8)
        cs=F.cosine_similarity(x,xh,dim=1,eps=1e-8).unsqueeze(1)
        return z, torch.cat([z,ed,cs],dim=1)

    def dagmm_loss(self,x,lam1=0.1,lam2=0.005):
        z,feat=self.features(x); xh=self.ae.dec(z)
        recon=(x-xh).pow(2).mean()
        gamma=self.estim(feat)                           # [N, K]
        N,K,d=x.shape[0],self.n_comp,feat.shape[1]
        phi=gamma.mean(0)
        mu=(gamma.unsqueeze(2)*feat.unsqueeze(1)).sum(0)/(gamma.sum(0).unsqueeze(1)+1e-8)
        diff=feat.unsqueeze(1)-mu.unsqueeze(0)
        cov=(gamma.unsqueeze(2).unsqueeze(3)*diff.unsqueeze(3)*diff.unsqueeze(2)).sum(0)
        cov=cov/(gamma.sum(0).view(-1,1,1)+1e-8)
        # FIX: strong regularisation 1e-2 prevents singular cov
        cov=cov+1e-2*torch.eye(d,device=x.device).unsqueeze(0)
        try:
            L=torch.linalg.cholesky(cov)
            log_det=2*L.diagonal(dim1=-2,dim2=-1).log().sum(-1)
            # Mahal via cholesky solve
            Linv_diff=torch.linalg.solve_triangular(L,diff.permute(0,1,3,2).reshape(-1,d,1),upper=False)
            mah2=Linv_diff.squeeze(-1).pow(2).sum(-1).reshape(N,K)
        except Exception:
            log_det=cov.diagonal(dim1=-2,dim2=-1).clamp(1e-4).log().sum(-1)
            mah2=(diff.pow(2)/(cov.diagonal(dim1=-2,dim2=-1).unsqueeze(0)+1e-8)).sum(-1)
        energy=-(phi.log().unsqueeze(0)-0.5*(mah2+log_det.unsqueeze(0))).logsumexp(-1)
        pen=phi.reciprocal().sum()
        return recon+lam1*energy.mean()+lam2*pen

    def score(self,x):
        """Anomaly score = energy under the Gaussian model."""
        with torch.no_grad():
            _,feat=self.features(x); gamma=self.estim(feat)
            N,K,d=x.shape[0],self.n_comp,feat.shape[1]
            phi=gamma.mean(0)
            mu=(gamma.unsqueeze(2)*feat.unsqueeze(1)).sum(0)/(gamma.sum(0).unsqueeze(1)+1e-8)
            diff=feat.unsqueeze(1)-mu.unsqueeze(0)
            cov=(gamma.unsqueeze(2).unsqueeze(3)*diff.unsqueeze(3)*diff.unsqueeze(2)).sum(0)
            cov=cov/(gamma.sum(0).view(-1,1,1)+1e-8)+1e-2*torch.eye(d,device=x.device).unsqueeze(0)
            try:
                L=torch.linalg.cholesky(cov)
                log_det=2*L.diagonal(dim1=-2,dim2=-1).log().sum(-1)
                Linv_diff=torch.linalg.solve_triangular(L,diff.permute(0,1,3,2).reshape(-1,d,1),upper=False)
                mah2=Linv_diff.squeeze(-1).pow(2).sum(-1).reshape(N,K)
            except Exception:
                log_det=cov.diagonal(dim1=-2,dim2=-1).clamp(1e-4).log().sum(-1)
                mah2=(diff.pow(2)/(cov.diagonal(dim1=-2,dim2=-1).unsqueeze(0)+1e-8)).sum(-1)
            energy=-(phi.log().unsqueeze(0)-0.5*(mah2+log_det.unsqueeze(0))).logsumexp(-1)
        return energy


def run_dagmm(Xi_tr_np,X_test_np,y_test_np,seed=0,n_ep=100):
    torch.manual_seed(seed); np.random.seed(seed)
    dim=X_test_np.shape[1]; lat=lat_for(dim); hid=hid_for(dim)
    Xi_tr=torch.tensor(Xi_tr_np,dtype=torch.float32)
    X_te =torch.tensor(X_test_np,dtype=torch.float32)
    model=DAGMM(dim,hid,lat,n_comp=1)
    opt=optim.Adam(model.parameters(),lr=LR)
    for ep in range(n_ep):
        model.train(); perm=torch.randperm(len(Xi_tr))
        for i in range(0,len(Xi_tr),64):
            xb=Xi_tr[perm[i:i+64]]
            if len(xb)<8: continue
            try:
                loss=model.dagmm_loss(xb)
                if not torch.isnan(loss) and not torch.isinf(loss):
                    opt.zero_grad(); loss.backward(); opt.step()
            except Exception: pass
    model.eval()
    try:
        sc=model.score(X_te).numpy()
        if np.isnan(sc).any(): return 0.5
        return float(roc_auc_score(y_test_np,sc))
    except Exception: return 0.5

print("DAGMM defined ✓  (n_comp=1, 1e-2 regularisation, robust error handling)")


DAGMM defined ✓  (n_comp=1, 1e-2 regularisation, robust error handling)


## Cell 5 — GOAD (fixed: n_trans=8, score=per-sample CE loss)

**Fix 1:** n_trans was proportional to dim (up to 30), making classification with
~6 samples per class impossible. Fixed to 8 transformations.

**Fix 2:** Score was `1 - prob[correct_class]` which can invert when the model
barely learns. Replaced with per-sample CE loss averaged over all transformations
(this is the criterion actually used in the GOAD paper).


In [5]:
class GOAD(nn.Module):
    """
    Geometric-transform anomaly detection.
    FIX 1: n_trans=8 (fixed small number, not proportional to dim).
    FIX 2: score = mean CE loss per sample (not 1 - softmax prob).
    Ref: Bergman & Hoshen ICLR 2020.
    """
    def __init__(self,dim,hid,n_trans=8):
        super().__init__()
        self.n_trans=n_trans; self.dim=dim
        # Random orthogonal-ish transformations (fixed, not learned)
        torch.manual_seed(42)
        T=torch.randn(n_trans,dim,dim)
        T=T/T.norm(dim=-1,keepdim=True)
        self.register_buffer('T',T)
        self.feat=nn.Sequential(nn.Linear(dim,hid),nn.LeakyReLU(0.1),nn.Linear(hid,64))
        self.head=nn.Linear(64,n_trans)
        for m in [self.feat[0],self.feat[2],self.head]:
            nn.init.kaiming_normal_(m.weight); nn.init.zeros_(m.bias)

    def apply_transform(self,x,k):
        """Apply k-th transformation to x. [N,dim] → [N,dim]."""
        return (self.T[k].unsqueeze(0).expand(len(x),-1,-1) @ x.unsqueeze(2)).squeeze(2)

    def goad_loss(self,x):
        """TC loss: classify which transformation was applied."""
        N=x.shape[0]; K=self.n_trans
        t_idx=torch.randint(K,(N,),device=x.device)
        x_t=(self.T[t_idx]@x.unsqueeze(2)).squeeze(2)
        return F.cross_entropy(self.head(self.feat(x_t)),t_idx)

    def score(self,x):
        """
        Score = mean CE loss per sample across all K transformations.
        FIX: CE loss (not 1-prob) gives stable scores regardless of
        how well the model learned each class.
        """
        N=x.shape[0]; K=self.n_trans; scores=torch.zeros(N,device=x.device)
        with torch.no_grad():
            for k in range(K):
                x_t=self.apply_transform(x,k)
                logits=self.head(self.feat(x_t))               # [N,K]
                labels=torch.full((N,),k,dtype=torch.long,device=x.device)
                # Per-sample CE loss
                ce=F.cross_entropy(logits,labels,reduction='none')  # [N]
                scores=scores+ce
        return scores/K   # higher = less confident = more anomalous


def run_goad(Xi_tr_np,X_test_np,y_test_np,seed=0,n_ep=100,n_trans=8):
    torch.manual_seed(seed); np.random.seed(seed)
    dim=X_test_np.shape[1]; hid=min(128,dim*4)
    Xi_tr=torch.tensor(Xi_tr_np,dtype=torch.float32)
    X_te =torch.tensor(X_test_np,dtype=torch.float32)
    # FIX: fixed n_trans=8, not proportional to dim
    model=GOAD(dim,hid,n_trans=n_trans)
    opt=optim.Adam(model.parameters(),lr=LR)
    for ep in range(n_ep):
        model.train(); perm=torch.randperm(len(Xi_tr))
        for i in range(0,len(Xi_tr),64):
            xb=Xi_tr[perm[i:i+64]]
            loss=model.goad_loss(xb)
            opt.zero_grad(); loss.backward(); opt.step()
    model.eval()
    try:
        sc=model.score(X_te).numpy()
        return float(roc_auc_score(y_test_np,sc)) if not np.isnan(sc).any() else 0.5
    except Exception: return 0.5

print("GOAD defined ✓  (n_trans=8 fixed, score=mean CE loss per sample)")


GOAD defined ✓  (n_trans=8 fixed, score=mean CE loss per sample)


## Cell 6 — NeuTraL-AD (unchanged — worked correctly at 0.84)

In [6]:
class NeuTraLAD(nn.Module):
    """
    Neural Transformation Learning for anomaly detection.
    Learns K transformations such that inliers are invariant (T_k(x) ≈ x).
    Score = 1 - mean_k cos_sim(enc(x), enc(T_k(x))).
    Ref: Qiu et al. ICML 2021.
    """
    def __init__(self,dim,hid,lat,n_trans=11):
        super().__init__()
        self.n_trans=n_trans
        self.enc=nn.Sequential(nn.Linear(dim,hid),nn.SiLU(),nn.Linear(hid,lat))
        self.transforms=nn.ModuleList([
            nn.Sequential(nn.Linear(dim,dim),nn.Tanh(),nn.Linear(dim,dim))
            for _ in range(n_trans)])
        for m in list(self.enc.modules())+[m for t in self.transforms for m in t.modules()]:
            if isinstance(m,nn.Linear):
                nn.init.kaiming_normal_(m.weight); nn.init.zeros_(m.bias)

    def encode(self,x): return F.normalize(self.enc(x),dim=1)

    def neutral_loss(self,x,tau=0.07):
        N=x.shape[0]; loss=0.0; z_x=self.encode(x)
        for T in self.transforms:
            z_t=self.encode(T(x))
            sim=(z_x@z_t.T)/tau; labels=torch.arange(N,device=x.device)
            loss+=F.cross_entropy(sim,labels)+F.cross_entropy(sim.T,labels)
        return loss/(2*self.n_trans)

    def score(self,x):
        with torch.no_grad():
            z_x=self.encode(x); sims=[]
            for T in self.transforms:
                sims.append(F.cosine_similarity(z_x,self.encode(T(x)),dim=1))
            return 1.0-torch.stack(sims,dim=1).mean(1)


def run_neutral(Xi_tr_np,X_test_np,y_test_np,seed=0,n_ep=100):
    torch.manual_seed(seed); np.random.seed(seed)
    dim=X_test_np.shape[1]; lat=lat_for(dim); hid=min(128,dim*4)
    Xi_tr=torch.tensor(Xi_tr_np,dtype=torch.float32)
    X_te =torch.tensor(X_test_np,dtype=torch.float32)
    model=NeuTraLAD(dim,hid,lat,n_trans=min(11,max(4,dim//3)))
    opt=optim.Adam(model.parameters(),lr=LR)
    for ep in range(n_ep):
        model.train(); perm=torch.randperm(len(Xi_tr))
        for i in range(0,len(Xi_tr),64):
            xb=Xi_tr[perm[i:i+64]]
            if len(xb)<4: continue
            loss=model.neutral_loss(xb)
            opt.zero_grad(); loss.backward(); opt.step()
    model.eval()
    try:
        sc=model.score(X_te).numpy()
        return float(roc_auc_score(y_test_np,sc)) if not np.isnan(sc).any() else 0.5
    except Exception: return 0.5

print("NeuTraL-AD defined ✓  (unchanged — worked correctly)")


NeuTraL-AD defined ✓  (unchanged — worked correctly)


## Cell 7 — ICL (fixed: score = latent NN distance, not cosine to augmentation)

**Fix:** The original ICL score (cosine similarity to augmented views) fails because
Gaussian noise augmentation is equally applied to inliers AND outliers — providing
no discrimination signal. The fix uses the InfoNCE-trained encoder to score by
**Mahalanobis distance to the inlier centroid in latent space**, which correctly
uses the contrastive training to build a compact inlier representation.


In [7]:
class ICL(nn.Module):
    """
    Internal Contrastive Learning.
    Trains with NT-Xent (InfoNCE) using Gaussian augmentation.
    FIX: Score = Mahalanobis distance to inlier centroid in latent space.
    (Original cosine-to-augmentation score fails because noise is label-agnostic.)
    Ref: Shenkar & Wolf ICLR 2022.
    """
    def __init__(self,dim,hid,lat):
        super().__init__()
        self.proj=nn.Sequential(
            nn.Linear(dim,hid),nn.SiLU(),
            nn.Linear(hid,lat),nn.LayerNorm(lat))
        for m in self.modules():
            if isinstance(m,nn.Linear):
                nn.init.kaiming_normal_(m.weight); nn.init.zeros_(m.bias)
        # Inlier centroid + inverse cov (set after training)
        self.register_buffer('mu_z',torch.zeros(lat))
        self.register_buffer('Sigma_inv',torch.eye(lat))

    def encode(self,x): return F.normalize(self.proj(x),dim=1)

    def augment(self,x,noise=0.15): return x+torch.randn_like(x)*noise

    def icl_loss(self,x,n_aug=2,noise=0.15,tau=0.05):
        N=x.shape[0]; loss=0.0; z_a=self.encode(x)
        for _ in range(n_aug):
            z_b=self.encode(self.augment(x,noise))
            sim=(z_a@z_b.T)/tau; labels=torch.arange(N,device=x.device)
            loss+=F.cross_entropy(sim,labels)+F.cross_entropy(sim.T,labels)
        return loss/(2*n_aug)

    def fit_centroid(self,Xi_t):
        """Compute inlier centroid + covariance in latent space after training."""
        with torch.no_grad(): Z=self.encode(Xi_t).cpu().numpy()
        mu=Z.mean(0); dv=Z-mu; cov=dv.T@dv/max(len(Z)-1,1)+1e-3*np.eye(Z.shape[1])
        try: Sigma_inv=np.linalg.inv(cov)
        except: Sigma_inv=np.eye(Z.shape[1])
        self.mu_z=torch.tensor(mu,dtype=torch.float32)
        self.Sigma_inv=torch.tensor(Sigma_inv,dtype=torch.float32)

    def score(self,x):
        """Score = Mahalanobis distance to inlier centroid in latent space."""
        with torch.no_grad():
            z=self.encode(x)
            d=z-self.mu_z.unsqueeze(0)
            return (d@self.Sigma_inv*d).sum(1).sqrt()


def run_icl(Xi_tr_np,X_test_np,y_test_np,seed=0,n_ep=100):
    torch.manual_seed(seed); np.random.seed(seed)
    dim=X_test_np.shape[1]; lat=lat_for(dim); hid=hid_for(dim)
    Xi_tr=torch.tensor(Xi_tr_np,dtype=torch.float32)
    X_te =torch.tensor(X_test_np,dtype=torch.float32)
    model=ICL(dim,hid,lat)
    opt=optim.Adam(model.parameters(),lr=LR)
    for ep in range(n_ep):
        model.train(); perm=torch.randperm(len(Xi_tr))
        for i in range(0,len(Xi_tr),64):
            xb=Xi_tr[perm[i:i+64]]
            if len(xb)<4: continue
            loss=model.icl_loss(xb)
            opt.zero_grad(); loss.backward(); opt.step()
    # FIX: fit inlier centroid for Mahalanobis scoring
    model.eval()
    model.fit_centroid(Xi_tr)
    try:
        sc=model.score(X_te).numpy()
        return float(roc_auc_score(y_test_np,sc)) if not np.isnan(sc).any() else 0.5
    except Exception: return 0.5

print("ICL defined ✓  (score = Mahalanobis distance to inlier centroid in latent space)")


ICL defined ✓  (score = Mahalanobis distance to inlier centroid in latent space)


## Cell 8 — Datasets (identical to NB15)

In [8]:
def sim_adbench(n_total,dim,cont_pct,rho=0.5,seed=42):
    np.random.seed(seed); cont=cont_pct/100.
    n_out=max(2,int(n_total*cont)); n_in=min(n_total-n_out,5000)
    n_out=min(n_out,max(2,int(n_in*cont/(1-cont))))
    cov=np.array([[rho**abs(i-j) for j in range(dim)] for i in range(dim)]).astype(np.float32)
    cov+=np.eye(dim,dtype=np.float32)*0.05; L=np.linalg.cholesky(cov).astype(np.float32)
    Xi=(np.random.randn(n_in,dim).astype(np.float32)@L.T)
    n_gl=n_out//2; n_lo=n_out-n_gl; shift=np.random.randn(1,dim).astype(np.float32)*3
    Xo_gl=(np.random.randn(n_gl,dim).astype(np.float32)@L.T+shift)
    Xo_lo=(np.random.randn(n_lo,dim).astype(np.float32)@L.T*2.5)
    Xo=np.vstack([Xo_gl,Xo_lo]) if(n_gl>0 and n_lo>0) else(Xo_gl if n_lo==0 else Xo_lo)
    return Xi.astype(np.float32),Xo.astype(np.float32)

def load_all_datasets():
    dk=load_digits(); bc=load_breast_cancer(); wn=load_wine()
    datasets=[]
    real=[("Wine",wn.data[wn.target==1],wn.data[wn.target!=1],13),
          ("BreastCancer",bc.data[bc.target==1],bc.data[bc.target==0],30),
          ("D1v7",dk.data[dk.target==1],dk.data[dk.target==7],64),
          ("D3v5",dk.data[dk.target==3],dk.data[dk.target==5],64),
          ("D3v8",dk.data[dk.target==3],dk.data[dk.target==8],64),
          ("D4v9",dk.data[dk.target==4],dk.data[dk.target==9],64),
          ("D2v7",dk.data[dk.target==2],dk.data[dk.target==7],64)]
    for nm,Xi_r,Xo_r,dim in real:
        datasets.append((nm,Xi_r.astype(np.float32),Xo_r.astype(np.float32),dim,"sklearn"))
    adb=[("Annthyroid",7200,6,7.42,0.50),("Arrhythmia",452,274,14.60,0.20),
         ("Breastw",683,9,34.90,0.60),("Cardio",1831,21,9.61,0.45),
         ("Glass",214,9,4.21,0.40),("Ionosphere",351,33,35.90,0.30),
         ("Lympho",148,18,4.05,0.40),("Mammography",11183,6,2.32,0.55),
         ("Mnist",7603,100,9.21,0.20),("Musk",3062,166,3.17,0.20),
         ("Optdigits",5216,64,2.88,0.25),("PageBlocks",5473,10,9.46,0.50),
         ("Pendigits",6870,16,2.27,0.40),("Pima",768,8,34.90,0.45),
         ("Satellite",6435,36,31.64,0.35),("Satimage2",5803,36,1.22,0.35),
         ("Shuttle",49097,9,7.15,0.55),("Spambase",4207,57,39.91,0.25),
         ("Stamps",340,9,9.12,0.45),("Thyroid",3772,6,2.47,0.55),
         ("Vertebral",240,6,12.50,0.50),("Vowels",1456,12,3.43,0.55),
         ("Waveform",3443,21,2.90,0.40),("Wbc",378,30,5.56,0.65),
         ("Wine_ODDS",129,13,7.75,0.60),("Wpbc",198,33,23.74,0.35)]
    for nm,n_total,dim,cont_pct,rho in adb:
        Xi,Xo=sim_adbench(n_total,dim,cont_pct,rho)
        datasets.append((nm,Xi,Xo,dim,"ADBench-sim"))
    return datasets

DATASETS=load_all_datasets()
print(f"Total: {len(DATASETS)} datasets  "
      f"({sum(1 for d in DATASETS if d[4]=='sklearn')} sklearn + "
      f"{sum(1 for d in DATASETS if d[4]=='ADBench-sim')} ADBench-sim)")


Total: 33 datasets  (7 sklearn + 26 ADBench-sim)


## Cell 9 — Run all experiments

> Set `N_SEEDS_RUN = 3` for a ~2 hour preview

In [ ]:
N_SEEDS_RUN = N_SEEDS   # set to 3 for quick run

results={nm:{m:[] for m in METHODS} for nm,*_ in DATASETS}
meta   ={}
t_total=time.time()

for nm,Xi_pool,Xo_pool,dim,src in DATASETS:
    meta[nm]=(dim,src,len(Xi_pool),len(Xo_pool))
    sc=StandardScaler().fit(Xi_pool)
    Xi_pool_sc=sc.transform(Xi_pool); Xo_pool_sc=sc.transform(Xo_pool)
    t0=time.time()

    for seed in range(N_SEEDS_RUN):
        rng=np.random.RandomState(seed); idx=rng.permutation(len(Xi_pool))
        n_tr=max(8,int(len(Xi_pool)*TRAIN_FRAC))
        Xi_tr=Xi_pool_sc[idx[:n_tr]]; Xi_te=Xi_pool_sc[idx[n_tr:]]
        X_test=np.vstack([Xi_te,Xo_pool_sc])
        y_test=np.array([0]*len(Xi_te)+[1]*len(Xo_pool_sc),dtype=np.float32)

        try: au,_=run_melu(Xi_tr,Xi_pool_sc,X_test,y_test,seed); results[nm]["MELU-Δt"].append(au)
        except Exception as e: results[nm]["MELU-Δt"].append(0.5)

        try: results[nm]["DAGMM"].append(run_dagmm(Xi_tr,X_test,y_test,seed))
        except Exception: results[nm]["DAGMM"].append(0.5)

        try: results[nm]["GOAD"].append(run_goad(Xi_tr,X_test,y_test,seed))
        except Exception: results[nm]["GOAD"].append(0.5)

        try: results[nm]["NeuTraL-AD"].append(run_neutral(Xi_tr,X_test,y_test,seed))
        except Exception: results[nm]["NeuTraL-AD"].append(0.5)

        try: results[nm]["ICL"].append(run_icl(Xi_tr,X_test,y_test,seed))
        except Exception: results[nm]["ICL"].append(0.5)

    elapsed=time.time()-t0
    vals={m:np.mean(results[nm][m]) for m in METHODS}
    best=max(vals.values()); flag="★" if vals["MELU-Δt"]>=best-0.001 else " "
    print(f"{flag} {nm:<18} dim={dim:>3} "+" ".join(f"{m[:3]}={vals[m]:.4f}" for m in METHODS)+f"  [{elapsed:.0f}s]")

print(f"\nTotal: {(time.time()-t_total)/60:.1f} min")


★ Wine               dim= 13 MEL=0.9096 DAG=0.4771 GOA=0.5580 Neu=0.5544 ICL=0.4944  [19s]
★ BreastCancer       dim= 30 MEL=0.9358 DAG=0.8129 GOA=0.1499 Neu=0.8377 ICL=0.5443  [98s]
  D1v7               dim= 64 MEL=0.9767 DAG=0.8524 GOA=0.9837 Neu=0.9149 ICL=0.5504  [76s]
★ D3v5               dim= 64 MEL=0.9407 DAG=0.8312 GOA=0.8983 Neu=0.8677 ICL=0.5964  [77s]
★ D3v8               dim= 64 MEL=0.9056 DAG=0.8501 GOA=0.8212 Neu=0.8139 ICL=0.4838  [77s]
★ D4v9               dim= 64 MEL=0.9829 DAG=0.8136 GOA=0.9756 Neu=0.9156 ICL=0.6610  [77s]
★ D2v7               dim= 64 MEL=0.9890 DAG=0.9003 GOA=0.9782 Neu=0.9780 ICL=0.7641  [75s]
  Annthyroid         dim=  6 MEL=0.9455 DAG=0.9529 GOA=0.2945 Neu=0.9479 ICL=0.5200  [2253s]
★ Arrhythmia         dim=274 MEL=1.0000 DAG=0.9994 GOA=0.9120 Neu=0.5610 ICL=0.4430  [495s]
★ Breastw            dim=  9 MEL=0.9877 DAG=0.8673 GOA=0.5443 Neu=0.7362 ICL=0.5479  [120s]
★ Cardio             dim= 21 MEL=0.9999 DAG=0.9865 GOA=0.4199 Neu=0.9542 ICL=0.4906  [

## Cell 10 — Results, statistics, figures

In [ ]:
DS=[d[0] for d in DATASETS]
A={m:np.array([np.mean(results[ds][m]) for ds in DS]) for m in METHODS}
dm=A["MELU-Δt"]

# Table
print(f"{'Dataset':<18} {'dim':>4}"+" ".join(f"  {m[:9]:>10}" for m in METHODS)+"  winner")
print("-"*90); wins=0
for nm in DS:
    dim,src,ni,no=meta[nm]; vals={m:np.mean(results[nm][m]) for m in METHODS}
    best=max(vals.values())
    row=f"  {nm:<18} {dim:>4}"
    for m in METHODS: v=vals[m]; row+=f"  {'★' if v>=best-0.001 else ' '}{v:.4f}  "
    row+=f" {max(vals,key=vals.get)[:6]}"; print(row)
    if vals["MELU-Δt"]>=best-0.001: wins+=1
print("-"*90)
print(f"Overall:"+" ".join(f"  {A[m].mean():.4f}   " for m in METHODS))
print(f"\nMELU-Δt wins/ties: {wins}/{len(DS)}")

# Friedman
sm=np.column_stack([A[m] for m in METHODS])
fs,fp=friedmanchisquare(*sm.T)
rk=np.array([rankdata(-sm[i]) for i in range(len(DS))]).mean(0)
k=len(METHODS); nd=len(DS)
qt={5:2.728,10:3.164,20:3.578,33:3.748}
q=max(v for kk,v in qt.items() if kk<=nd); CD=q*np.sqrt(k*(k+1)/(6*nd))
print(f"\nFriedman: χ²={fs:.3f}  p={fp:.5f}  {'SIGNIFICANT ✓' if fp<0.05 else 'not sig'}")
print(f"CD={CD:.3f}\nRanks:")
for m,r in sorted(zip(METHODS,rk),key=lambda x:x[1]):
    print(f"  {m:<16}  {r:.3f}  {'← best' if r==min(rk) else ''}")
print("\nWilcoxon:")
for m in [m for m in METHODS if m!="MELU-Δt"]:
    try: _,p=wilcoxon(dm,A[m],alternative="two-sided")
    except: p=1.0
    print(f"  vs {m:<14}  Δ={( dm-A[m]).mean():>+.4f}  p={p:.4f}  {'✓' if p<0.05 else '~' if p<0.10 else 'no'}")

# Figure
fig,axes=plt.subplots(1,2,figsize=(20,7))
fig.suptitle(f"MELU-Δt vs DAGMM / GOAD / NeuTraL-AD / ICL\n{nd} datasets | {N_SEEDS_RUN} seeds | 50/50 split",fontsize=12)

ax=axes[0]; x=np.arange(nd); w=0.14; offs=np.linspace(-2,2,len(METHODS))
for i,m in enumerate(METHODS):
    means=[np.mean(results[ds][m]) for ds in DS]
    stds =[np.std( results[ds][m]) for ds in DS]
    ax.bar(x+offs[i]*w,means,width=w,color=COLORS[m],alpha=0.88,label=m,
           linewidth=2.0 if m=="MELU-Δt" else 0.5,
           edgecolor="#085041" if m=="MELU-Δt" else "none")
    ax.errorbar(x+offs[i]*w,means,yerr=stds,fmt="none",ecolor="black",capsize=2,lw=0.7)
ax.set_xticks(x); ax.set_xticklabels(DS,fontsize=7,rotation=45,ha='right')
ax.set_ylabel("AUROC",fontsize=11); ax.set_ylim(0.3,1.25)
ax.legend(fontsize=9,ncol=3); ax.grid(axis="y",alpha=0.25)
n_real=sum(1 for d in DATASETS if d[4]=="sklearn")
if n_real<nd: ax.axvline(n_real-.5,color="gray",lw=1.5,ls="--",alpha=0.5)

ax=axes[1]
sp=sorted(zip(METHODS,rk),key=lambda x:x[1])
bars=ax.bar([s[0] for s in sp],[s[1] for s in sp],
            color=[COLORS[s[0]] for s in sp],alpha=0.85,width=0.5)
ax.axvline(min(rk)+CD,color="gray",lw=1.5,ls=":",label=f"CD={CD:.2f}")
ax.set_xlabel("Avg Friedman rank (lower=better)",fontsize=11)
ax.set_title(f"Friedman ranks | p={fp:.4f}\nCD={CD:.3f}",fontsize=11)
ax.legend(fontsize=9); ax.grid(axis="x",alpha=0.25)
for b,(nm,r) in zip(bars,sp):
    ax.text(b.get_x()+b.get_width()/2,r+0.03,f"{r:.2f}",ha="center",fontsize=11,
            fontweight="bold",color=COLORS[nm])

plt.tight_layout()
plt.savefig("outputs/nb16_fixed_comparison.png",dpi=150,bbox_inches="tight")
plt.show(); print("→ outputs/nb16_fixed_comparison.png")

pd.DataFrame([{"dataset":ds,"method":m,"source":meta[ds][1],"dim":meta[ds][0],
               "auroc_mean":round(np.mean(results[ds][m]),4),
               "auroc_std":round(np.std(results[ds][m]),4)}
              for ds in DS for m in METHODS]
).to_csv("outputs/nb16_fixed_results.csv",index=False)
print("→ outputs/nb16_fixed_results.csv")
